# MLP Experiment — Phase 0 Analysis

Torch MLP (4-16-16-3), iris, 5-fold, 10 seeds, `tc=2`, `feat=2`.
Each fold fits **one** model attacked by **both** HopSkipJump (black-box) and DeepFool
(white-box), so the attack contrast shares identical weights.

Spread is inverse-transformed back to raw iris units, then normalised **per seed** to that
seed's own clean baseline. **1.0 = no effect.**

Written narrative: `FINDINGS_phase0.md` · contract: `PLAN.md` · text tables: `analyze.py`

In [ ]:
from pathlib import Path
import numpy as np, polars as pl
import matplotlib.pyplot as plt

HERE = Path.cwd() if Path.cwd().name == 'mlp_experiment' else Path.cwd() / 'mlp_experiment'
PLOTS = HERE / 'plots'; PLOTS.mkdir(exist_ok=True)
DF = pl.read_parquet(HERE / 'results.parquet')

# Tufte-ish house style, matching generate_figures_v2.py (the repo's preferred figures)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 200, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.25, 'legend.frameon': False})
C = {'hsj': '#1e5eff', 'deepfool': '#e8590c'}          # blue = black-box, orange = white-box
LS = {'before_split': '--', 'train_only': '-'}

def ci95(v):
    """mean, half-width of the 95% CI (NaN-safe)."""
    v = np.asarray([x for x in v if np.isfinite(x)], float)
    if len(v) < 2:
        return (float(v[0]) if len(v) else np.nan), np.nan
    return float(v.mean()), float(1.96 * v.std(ddof=1) / np.sqrt(len(v)))

def normalise(df, baseline):
    """norm = mean_dist / that (attack, seed)'s clean baseline — paired, so 10 seeds suffice."""
    b = baseline.group_by(['attack', 'seed']).agg(pl.col('mean_dist').mean().alias('_b'))
    return df.join(b, on=['attack', 'seed'], how='left').with_columns(
        (pl.col('mean_dist') / pl.col('_b')).alias('norm'))

CG = DF.filter(pl.col('defect') == 'coverage_gap')
OL = DF.filter(pl.col('defect') == 'outlier')
CG_N = normalise(CG.filter(pl.col('bias') > 0), CG.filter(pl.col('bias') == 0.0))
OL_N = normalise(OL.filter(pl.col('n_out') > 0), OL.filter(pl.col('n_out') == 0))
print(f'{len(DF)} rows / {len(DF)//2} cells, {int(DF["mean_dist"].is_nan().sum())} NaN')

## F1 — the headline: "accuracy stays flat" is a test-set artifact

Same defect, same model, same weights. Only **when** the deletion happens changes.

Under `before_split` the deleted region leaves the *test* set too, so the model is scored only
on the easy remainder — accuracy does not stay flat, it **rises to 0.999** and minority recall
reaches **1.000**. Under the correct `train_only` protocol both collapse.

This is the controlled version of the 2026-08-11 imbalance finding, which changed protocol *and*
tree depth together. An MLP has no depth knob, so the protocol is isolated.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True)
biases = sorted(b for b in CG['bias'].unique() if b > 0)

for ax, metric, title in [(axes[0], 'vacc', 'Test accuracy'),
                          (axes[1], 'min_recall', 'Minority-class recall (virginica)')]:
    for proto in ['before_split', 'train_only']:
        sub = CG.filter((pl.col('protocol') == proto) & (pl.col('bias') > 0) &
                        (pl.col('attack') == 'hsj'))          # metric is attack-independent
        m = [ci95(sub.filter(pl.col('bias') == b)[metric].to_list()) for b in biases]
        mu, hw = np.array([x[0] for x in m]), np.array([x[1] for x in m])
        ax.plot(biases, mu, LS[proto], marker='o', ms=4,
                color='#c92a2a' if proto == 'train_only' else '#868e96',
                label='train_only (correct)' if proto == 'train_only' else 'before_split (grids 1-3)')
        ax.fill_between(biases, mu - hw, mu + hw, alpha=0.15,
                        color='#c92a2a' if proto == 'train_only' else '#868e96')
    ax.set_title(title); ax.set_xlabel('bias (fraction of class deleted)'); ax.set_ylim(0, 1.05)
axes[0].set_ylabel('score'); axes[0].legend(loc='lower left')
fig.suptitle('The defect is only invisible to accuracy when you also delete it from the test set',
             y=1.02, fontsize=11)
fig.tight_layout(); fig.savefig(PLOTS / 'f1_protocol_accuracy.png', bbox_inches='tight')
plt.show()

## F2 / F3 — the spread signal itself, and the bias-0.9 HSJ drop

The signal does **not** depend on the confound: under `train_only` both attacks still rise
monotonically (HSJ d = +2.81).

Watch the dashed blue line: HSJ/`before_split` reproduces the **non-monotonic drop at bias 0.9**
seen on Tree+HSJ, and it **disappears** under `train_only` (solid blue) — confirming test-set
depletion as the mechanism, which the repo had flagged as asserted but unverified.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
for attack in ['hsj', 'deepfool']:
    for proto in ['before_split', 'train_only']:
        sub = CG_N.filter((pl.col('attack') == attack) & (pl.col('protocol') == proto))
        m = [ci95(sub.filter(pl.col('bias') == b)['norm'].to_list()) for b in biases]
        mu, hw = np.array([x[0] for x in m]), np.array([x[1] for x in m])
        ax.errorbar(biases, mu, yerr=hw, fmt=LS[proto], marker='o', ms=4, capsize=3,
                    color=C[attack], alpha=1.0 if proto == 'train_only' else 0.55,
                    label=f'{attack} / {proto}')
ax.axhline(1.0, color='k', lw=0.8, ls=':')
ax.annotate('no effect', xy=(0.9, 1.0), xytext=(0.78, 0.955), fontsize=8, color='k')
ax.annotate('HSJ drop:\ntest set depleted', xy=(0.9, 1.02), xytext=(0.62, 1.24), fontsize=8,
            color=C['hsj'], ha='center',
            arrowprops=dict(arrowstyle='->', color=C['hsj'], lw=0.8))
ax.set_xlabel('bias (fraction of class deleted)')
ax.set_ylabel('normalised adversarial spread  (1.0 = clean baseline)')
ax.set_title('Coverage gap: the geometric signal survives a global smooth boundary')
ax.legend(loc='upper left', fontsize=8)
fig.tight_layout(); fig.savefig(PLOTS / 'f2_coverage_gap_spread.png', bbox_inches='tight')
plt.show()

## F4 — H6 refuted: a global boundary does *not* see outward outliers

Pre-registered prediction: an MLP's globally-parameterised boundary should be dragged by a far
same-labeled point, so `outward` should fire where every local model is null.

It doesn't — and neither does `toward`. So rbf **locality** does not explain the SVM null, and
the tree's 1.25–1.36× now looks specific to *axis-aligned partitioning under white-box attack*.

In [ ]:
# Reference values from Thread A / B findings, same strong cell (k=8, n_out=10)
REF = [('tree d3\n(DTA, WB)', 1.36, 1.01), ('tree d10\n(DTA, WB)', 1.25, 1.00),
       ('rbf-SVM\n(HSJ)', 0.98, 1.05), ('RandomForest\n(HSJ)', 0.92, 1.04)]
strong = OL_N.filter((pl.col('k') == 8.0) & (pl.col('n_out') == 10))
mlp = []
for a, lbl in [('hsj', 'MLP\n(HSJ)'), ('deepfool', 'MLP\n(DeepFool, WB)')]:
    vals = {d: ci95(strong.filter((pl.col('attack') == a) & (pl.col('direction') == d))['norm'].to_list())
            for d in ['toward', 'outward']}
    mlp.append((lbl, vals['toward'], vals['outward']))

labels = [r[0] for r in REF] + [m[0] for m in mlp]
tw = np.array([r[1] for r in REF] + [m[1][0] for m in mlp])
ow = np.array([r[2] for r in REF] + [m[2][0] for m in mlp])
tw_e = np.array([m[1][1] for m in mlp]); ow_e = np.array([m[2][1] for m in mlp])

n = len(REF); x = np.arange(len(labels)); w = 0.38
fig, ax = plt.subplots(figsize=(9.5, 4.2))
# The reference bars are POINT ESTIMATES quoted from the prior findings docs, which do not
# publish CIs. Drawn faded and WITHOUT error bars, so they are not read as more precise than
# this thread's 10-seed measurements -- zero-length caps would imply exactly that.
ax.bar(x[:n] - w/2, tw[:n], w, color='#2f9e44', alpha=0.45, label='toward')
ax.bar(x[:n] + w/2, ow[:n], w, color='#adb5bd', alpha=0.45, label='outward')
ax.bar(x[n:] - w/2, tw[n:], w, yerr=tw_e, capsize=3, color='#2f9e44')
ax.bar(x[n:] + w/2, ow[n:], w, yerr=ow_e, capsize=3, color='#adb5bd')
ax.axhline(1.0, color='k', lw=0.8, ls=':')
ax.axvline(n - 0.5, color='k', lw=0.8, alpha=0.3)
ax.text(n - 0.45, 1.44, 'this thread (10 seeds, 95% CI)', fontsize=9, style='italic')
ax.text(-0.4, 1.44, 'prior findings (point estimates, no CIs published)',
        fontsize=9, style='italic', color='#666')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8.5)
ax.set_ylabel('normalised spread  (1.0 = no effect)'); ax.set_ylim(0.8, 1.5)
ax.set_title('Outlier, strong cell (k=8, n_out=10): only the tree reacts, and only to `toward`')
ax.legend(loc='upper right')
fig.tight_layout(); fig.savefig(PLOTS / 'f4_outlier_model_comparison.png', bbox_inches='tight')
plt.show()

## H7 — no dose-response

Trees only react at `k >= 6` (~6 sigma, i.e. data-entry-error scale). The open question was
whether a more sensitive model reacts at *realistic* 2–3 sigma. The MLP reacts at **no** distance,
so the answer is negative for this model family.

In [ ]:
ks = sorted(OL_N['k'].unique())
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, d in zip(axes, ['toward', 'outward']):
    for attack in ['hsj', 'deepfool']:
        sub = OL_N.filter((pl.col('attack') == attack) & (pl.col('direction') == d) &
                          (pl.col('n_out') == 10))
        m = [ci95(sub.filter(pl.col('k') == k)['norm'].to_list()) for k in ks]
        mu, hw = np.array([x[0] for x in m]), np.array([x[1] for x in m])
        ax.errorbar(ks, mu, yerr=hw, marker='o', ms=4, capsize=3, color=C[attack], label=attack)
    ax.axhline(1.0, color='k', lw=0.8, ls=':')
    ax.axvspan(2, 3, color='#ffd43b', alpha=0.18)
    ax.text(2.5, 1.30, 'realistic\noutliers', fontsize=7.5, ha='center', color='#b8860b')
    ax.set_title(f'direction = {d}'); ax.set_xlabel('k  (distance in class sigma)')
axes[0].set_ylabel('normalised spread'); axes[0].legend(loc='upper left', fontsize=8)
fig.suptitle('No dose-response at any distance (n_out=10)', y=1.03, fontsize=11)
fig.tight_layout(); fig.savefig(PLOTS / 'h7_outlier_dose_response.png', bbox_inches='tight')
plt.show()

---
### Figures written to `plots/`

| file | shows |
|---|---|
| `f1_protocol_accuracy.png` | **the headline** — accuracy rises before-split, collapses train-only |
| `f2_coverage_gap_spread.png` | spread signal survives; the HSJ bias-0.9 drop is protocol-caused |
| `f4_outlier_model_comparison.png` | H6 refuted, MLP against all four previously-tested models |
| `h7_outlier_dose_response.png` | no dose-response at any distance |